# Phase 9 — MLOps: tracking, serving, drift

MLflow experiment tracking, a FastAPI scoring endpoint over the saved pipeline, and PSI drift monitoring.

In [1]:
import sys; sys.path.append('..')


## Drift monitoring (PSI)

In [2]:
import numpy as np
from src.monitor import psi, psi_verdict
rng=np.random.default_rng(0)
ref=rng.normal(size=5000)
print('no drift :', round(psi(ref, rng.normal(size=5000)),4), '->', psi_verdict(psi(ref, rng.normal(size=5000))))
print('big drift:', round(psi(ref, rng.normal(3,1,5000)),4), '->', psi_verdict(psi(ref, rng.normal(3,1,5000))))


no drift : 0.0055 -> stable
big drift: 7.446 -> major shift — retrain


## Serving (FastAPI) — score a transaction over the saved pipeline

The served artifact is the FULL preprocessing+model pipeline, so there is no train/serve skew.

In [3]:
from fastapi.testclient import TestClient
from src.serve import app
c = TestClient(app)
print('health:', c.get('/health').json())
print(c.post('/predict', json={'amount':950.0,'log_amount':6.86,'hour':3,'spend_prior':40.0,'count_prior':2,'txns_prior_lifetime':9}).json())


/sessions/awesome-friendly-fermi/.local/lib/python3.10/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


health: {'status': 'ok'}
{'fraud_probability': 0.9630205992332902, 'decision': 'review', 'threshold': 0.42}


## Experiment tracking (MLflow)

`python -m src.track` logs params, metrics, and the pipeline artifact to a SQLite-backed MLflow store. See `reports/monitoring_runbook.md` for the retraining policy.